In [3]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

TRAIN = ROOT / "data/raw/train"
VALIDATION = ROOT / "data/raw/validation"
TEST = ROOT / "data/raw/test"

transactions = pd.read_parquet(TRAIN / "transactions.parquet")
transfers = pd.read_parquet(TRAIN / "transfers.parquet")
blocks = pd.read_parquet(TRAIN / "blocks.parquet")
rates = pd.read_csv(TRAIN / "market_rates.csv")
labels = pd.read_csv(TRAIN / "labels.csv")

for name, df in {
    "transactions": transactions,
    "transfers": transfers,
    "blocks": blocks,
    "rates": rates,
    "labels": labels
}.items():
    print(f"\n{name}: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isna().sum().sum())


transactions: (40131, 26)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'TRANSACTION_INDEX', 'ACCOUNT', 'TRANSACTION_TYPE', 'SOURCE_FEE', 'FEE', 'SEQUENCE', 'ACCOUNT_TRANSACTION_ID', 'FLAGS', 'LAST_LEDGER_SEQUENCE', 'MEMOS', 'SIGNERS', 'SOURCE_TAG', 'SIGNING_PUB_KEY', 'TICKET_SEQUENCE', 'TXN_SIGNATURE', 'TRANSACTION_RESULT', 'DELIVERED_AMOUNT', 'SUCCESS', 'EXTRA_TRANSACTION_FIELDS', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 280917

transfers: (60557, 16)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_INDEX', 'TRANSACTION_HASH', 'TRANSFER_TYPE', 'TRANSFER_INDEX', 'FROM_ADDRESS', 'TO_ADDRESS', 'TOKEN_ADDRESS', 'SYMBOL', 'SOURCE_VALUE', 'VALUE', 'DATE_CREATED', 'DATE_UPDATED']
Missing values: 0

blocks: (151200, 17)
Columns: ['LEDGER_INDEX', 'TIMESTAMP', 'DATETIME', 'LEDGER_HASH', 'TRANSACTION_HASH', 'VALIDATED', 'ACCOUNT_HASH', 'TOTAL_COINS', 'CLOSE_FLAGS', 'CLOSE_TIME', 'CLOSE_TIME_HUMAN', 'CLOSE_TIME_RESOLUT

In [4]:
# 1. Missing values per column
print("=== TRANSACTIONS: MISSING VALUES ===")
print(
    transactions.isna().sum()
    .sort_values(ascending=False)
    .to_string()
)

print("\n=== LABELS: MISSING VALUES ===")
print(labels.isna().sum().to_string())

# 2. Duplicate checks
print("\n=== DUPLICATES ===")
print("Transactions duplicate hashes:",
      transactions["TRANSACTION_HASH"].duplicated().sum())

print("Labels duplicate hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("Transfers duplicate rows:",
      transfers.duplicated().sum())

# 3. Target distribution
print("\n=== TARGET DISTRIBUTION ===")
print(labels["is_suspicious"].value_counts(dropna=False))
print(labels["is_suspicious"].value_counts(normalize=True, dropna=False))

# 4. Transaction types
print("\n=== TRANSACTION TYPES ===")
print(transactions["TRANSACTION_TYPE"].value_counts(dropna=False))

# 5. Transfer counts per transaction
print("\n=== TRANSFERS PER TRANSACTION ===")
print(transfers.groupby("TRANSACTION_HASH").size().describe())

=== TRANSACTIONS: MISSING VALUES ===
MEMOS                       40131
TXN_SIGNATURE               40131
TICKET_SEQUENCE             40131
SIGNING_PUB_KEY             40131
SOURCE_TAG                  40131
ACCOUNT_TRANSACTION_ID      40131
SIGNERS                     40131
LEDGER_INDEX                    0
DATE_CREATED                    0
EXTRA_TRANSACTION_FIELDS        0
SUCCESS                         0
DELIVERED_AMOUNT                0
TRANSACTION_RESULT              0
LAST_LEDGER_SEQUENCE            0
TIMESTAMP                       0
FLAGS                           0
SEQUENCE                        0
FEE                             0
SOURCE_FEE                      0
TRANSACTION_TYPE                0
ACCOUNT                         0
TRANSACTION_INDEX               0
TRANSACTION_HASH                0
LEDGER_HASH                     0
DATETIME                        0
DATE_UPDATED                    0

=== LABELS: MISSING VALUES ===
TRANSACTION_HASH        0
is_suspicious        

In [5]:
columns_to_drop = [
    "MEMOS",
    "TXN_SIGNATURE",
    "TICKET_SEQUENCE",
    "SIGNING_PUB_KEY",
    "SOURCE_TAG",
    "ACCOUNT_TRANSACTION_ID",
    "SIGNERS"
]

transactions_clean = transactions.drop(
    columns=columns_to_drop
).copy()

print("Before:", transactions.shape)
print("After:", transactions_clean.shape)
print("Missing values:", transactions_clean.isna().sum().sum())

Before: (40131, 26)
After: (40131, 19)
Missing values: 0


We check for duplicate transaction hashes and missing target labels to ensure data consistency. 

In [6]:
print("=== DUPLICATES ===")

print("Duplicate transaction hashes:",
      transactions_clean["TRANSACTION_HASH"].duplicated().sum())

print("Duplicate label hashes:",
      labels["TRANSACTION_HASH"].duplicated().sum())

print("\n=== TARGET ===")

print("Missing target values:",
      labels["is_suspicious"].isna().sum())

print("\nTarget distribution:")
print(labels["is_suspicious"].value_counts())

print("\nTarget percentages:")
print(labels["is_suspicious"].value_counts(normalize=True) * 100)

=== DUPLICATES ===
Duplicate transaction hashes: 0
Duplicate label hashes: 0

=== TARGET ===
Missing target values: 0

Target distribution:
is_suspicious
0    38881
1     1250
Name: count, dtype: int64

Target percentages:
is_suspicious
0    96.885201
1     3.114799
Name: proportion, dtype: float64


Some transactions contain multiple transfers. We aggregate these records by transaction hash to create four features: the number of transfers, unique senders, unique receivers, and unique tokens. This gives us additional information about each transaction without creating duplicate rows.

In [7]:
transfer_features = transfers.groupby("TRANSACTION_HASH").agg(
    transfer_count=("TRANSFER_INDEX", "count"),
    unique_senders=("FROM_ADDRESS", "nunique"),
    unique_receivers=("TO_ADDRESS", "nunique"),
    unique_tokens=("SYMBOL", "nunique")
).reset_index()

print("Shape:", transfer_features.shape)
display(transfer_features.head())

Shape: (40131, 5)


,TRANSACTION_HASH,transfer_count,unique_senders,unique_receivers,unique_tokens
0,00017BB7FD900622F10386CAF5F82BFD5575881005E331...,1,1,1,1
1,00028A6E2260BEDF874174AD4DBC96B139F85A4E9ABD9D...,3,3,3,3
2,00032386B56944BF3295207787EAF0A7C3C85947494B66...,1,1,1,1
3,000995ECC9CC8783B4C0612EA4AEE4A428B848DA7952C1...,3,3,3,3
4,0009B43313655466F274B66204B71902F616989ECC9138...,3,3,3,3


We merge the aggregated transfer features with the cleaned transactions using TRANSACTION_HASH. A left join ensures that all original transactions are preserved. For transactions without matching transfers, the new count features are set to zero.

In [8]:
# Merge transaction data with aggregated transfer features
train_features = transactions_clean.merge(
    transfer_features,
    on="TRANSACTION_HASH",
    how="left",
    validate="one_to_one"
)

# Transactions without transfer records receive zero counts
feature_cols = [
    "transfer_count",
    "unique_senders",
    "unique_receivers",
    "unique_tokens"
]

train_features[feature_cols] = (
    train_features[feature_cols].fillna(0).astype(int)
)

print("Original transactions:", len(transactions_clean))
print("After merge:", len(train_features))
print("Duplicate hashes:",
      train_features["TRANSACTION_HASH"].duplicated().sum())

print("\nMissing values:")
print(train_features.isna().sum().sum())

print("\nNew shape:", train_features.shape)

display(train_features[
    ["TRANSACTION_HASH"] + feature_cols
].head())

Original transactions: 40131
After merge: 40131
Duplicate hashes: 0

Missing values:
0

New shape: (40131, 23)


,TRANSACTION_HASH,transfer_count,unique_senders,unique_receivers,unique_tokens
0,954736A6E1BB5A3433EB916D8221F2DF5938A521EA2222...,1,1,1,1
1,C93CF387C2148608329ED89420105EFD883020CDBFDFAE...,1,1,1,1
2,E46BFC5EC409B486E1C97F20BCB62AE96432B20937D265...,1,1,1,1
3,E7CAB61BBDAC502147EF393C7EA93A2A937DA3F01623E2...,3,3,3,3
4,E9549020D2717DB401CA4C30BB82500758ABF100600D9C...,2,2,2,2


We add is_suspicious to the training dataset, indicating whether each transaction is suspicious (1) or normal (0). We exclude other label-related metadata to avoid introducing target leakage into the model.

In [9]:
train_features = train_features.merge(
    labels[["TRANSACTION_HASH", "is_suspicious"]],
    on="TRANSACTION_HASH",
    how="left",
    validate="one_to_one"
)

print("Shape:", train_features.shape)
print("Missing labels:", train_features["is_suspicious"].isna().sum())

Shape: (40131, 24)
Missing labels: 0


Preparing Features for Machine Learning

We inspect the data types and sample values to identify which columns can be used directly for training and which require further transformation or encoding.

In [10]:
print("=== COLUMN TYPES ===")
print(train_features.dtypes.to_string())

print("\n=== SAMPLE DATA ===")
display(train_features.head(3))

=== COLUMN TYPES ===
LEDGER_INDEX                             object
TIMESTAMP                                object
DATETIME                    datetime64[ms, UTC]
LEDGER_HASH                                 str
TRANSACTION_HASH                            str
TRANSACTION_INDEX                        object
ACCOUNT                                     str
TRANSACTION_TYPE                            str
SOURCE_FEE                               object
FEE                                     float64
SEQUENCE                                 object
FLAGS                                    object
LAST_LEDGER_SEQUENCE                     object
TRANSACTION_RESULT                          str
DELIVERED_AMOUNT                            str
SUCCESS                                    bool
EXTRA_TRANSACTION_FIELDS                    str
DATE_CREATED                datetime64[ms, UTC]
DATE_UPDATED                datetime64[ms, UTC]
transfer_count                            int64
unique_senders     

,LEDGER_INDEX,TIMESTAMP,DATETIME,LEDGER_HASH,TRANSACTION_HASH,TRANSACTION_INDEX,ACCOUNT,TRANSACTION_TYPE,SOURCE_FEE,FEE,...,DELIVERED_AMOUNT,SUCCESS,EXTRA_TRANSACTION_FIELDS,DATE_CREATED,DATE_UPDATED,transfer_count,unique_senders,unique_receivers,unique_tokens,is_suspicious
0,1124529,1788718916,2026-09-06 18:21:56+00:00,6F7094E1DDA27405855AB2CE4E97A555E0B372D45625B3...,954736A6E1BB5A3433EB916D8221F2DF5938A521EA2222...,0,r3CWFt1SXiykW92LwKU19CvfTFcFRnbnDY,Payment,10,0.000010,...,"""1167184470""",True,"{""Amount"":""1167184470"",""DeliverMax"":""116718447...",2026-09-06 18:21:56+00:00,2026-09-06 18:21:56+00:00,1,1,1,1,0
1,1136375,1788766300,2026-09-07 07:31:40+00:00,6760D562E18F762A5068E23BAC9CDE21AE43D11C44E466...,C93CF387C2148608329ED89420105EFD883020CDBFDFAE...,1,rEFG2WG1W8bHQhHp2P1rcxRBMxmz9C52B7,Payment,15,0.000015,...,"""486435229""",True,"{""Amount"":""486435229"",""DeliverMax"":""486435229""...",2026-09-07 07:31:40+00:00,2026-09-07 07:31:40+00:00,1,1,1,1,0
2,1129067,1788737068,2026-09-06 23:24:28+00:00,FCBE03345724C312C98168B9F2C4C0FA2B2A181B6FBFA3...,E46BFC5EC409B486E1C97F20BCB62AE96432B20937D265...,1,rJxqpGRZ2bJtB1Q8HYCV9iPon9HpvJugZm,Payment,15,0.000015,...,"{""currency"": ""EUR"", ""issuer"": ""rLeMq5iFyEhgHLf...",True,"{""Amount"":{""currency"":""EUR"",""issuer"":""rLeMq5iF...",2026-09-06 23:24:28+00:00,2026-09-06 23:24:28+00:00,1,1,1,1,0
